# 04a · Main tables (C1, C2)

**Purpose.** C1: representation × head tables of macro F1 (mean ± std), one per dataset and task (4 tables), plus the end to end reference row and confusion matrices of the best model per representation. C2: compare the ranking of R1, R2, R3 between GasWindTunnel and EverydayScents.

**Inputs**
- `results/01a`, `01b`, `02b`, `02c`, `03a`, `03b`, `03c` csv files

**Outputs**
- `report/tables/`: 4 tables (3 representations + reference × 4 heads)
- `report/figures/`: confusion matrices, ranking comparison

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from sklearn.metrics import confusion_matrix

TABLES = ROOT / "report/tables"
FIG = ROOT / "report/figures"
TABLES.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / "results"
MAIN = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
        "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
missing = [n for n in MAIN if not (RESULTS / f"{n}.csv").exists()]
assert not missing, f"run these notebooks first: {missing}"
scores = pd.concat([pd.read_csv(RESULTS / f"{n}.csv") for n in MAIN], ignore_index=True)
REPS = ["R1_raw", "R2_handcrafted", "R3_encoder"]
HEADS = ["logreg", "svm", "rf", "mlp"]

## C1: representation × head, macro F1 (mean ± std over 3 seeds × 5 folds)
One table per dataset and task. The end to end CNN is a single reference value.

In [ ]:
def c1_table(dataset, task, metric="macro_f1"):
    s = scores[(scores["dataset"] == dataset) & (scores["task"] == task)]
    g = s.groupby(["rep", "head"])[metric].agg(["mean", "std"])
    cell = g.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1).unstack("head")
    table = cell.reindex(index=REPS, columns=HEADS)
    ref = g.loc[("R3_end_to_end", "end_to_end")] if ("R3_end_to_end", "end_to_end") in g.index else None
    table.loc["Reference: end to end CNN"] = [f"{ref['mean']:.3f} ± {ref['std']:.3f}" if ref is not None else "", "", "", ""]
    return table


for dataset in ["gwt", "es"]:
    for task in ["multi", "binary"]:
        t = c1_table(dataset, task)
        t.to_csv(TABLES / f"C1_{dataset}_{task}.csv")
        print(f"{dataset} / {task}")
        display(t)

## C2: lab vs real world, ranking of the three representations
Score of a representation = its best head (highest mean macro F1).

In [ ]:
best = (scores[scores["rep"].isin(REPS)]
        .groupby(["dataset", "task", "rep", "head"])["macro_f1"].mean()
        .groupby(["dataset", "task", "rep"]).max().rename("best_head_f1").reset_index())
best["rank"] = best.groupby(["dataset", "task"])["best_head_f1"].rank(ascending=False).astype(int)
c2 = best.pivot_table(index="rep", columns=["task", "dataset"], values=["best_head_f1", "rank"])
c2.to_csv(TABLES / "C2_ranking.csv")
c2

## Confusion matrices: best head per representation, multiclass task
Predictions of all test folds of all seeds, read from the `_preds` files (nothing is refitted).

In [ ]:
preds = pd.concat([pd.read_csv(RESULTS / f"{n}_preds.csv") for n in MAIN], ignore_index=True)
classes = {d: json.load(open(ROOT / f"data/processed/{f}/info.json"))["classes"]
           for d, f in [("gwt", "gas_wind_tunnel"), ("es", "everyday_scents")]}

for dataset in ["gwt", "es"]:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, rep in zip(axes, REPS + ["R3_end_to_end"]):
        if rep == "R3_end_to_end":
            head = "end_to_end"
        else:
            means = scores[(scores["dataset"] == dataset) & (scores["task"] == "multi") & (scores["rep"] == rep)]
            head = means.groupby("head")["macro_f1"].mean().idxmax()
        p = preds[(preds["dataset"] == dataset) & (preds["task"] == "multi") & (preds["rep"] == rep) & (preds["head"] == head)]
        K = len(classes[dataset])
        cm = confusion_matrix(p["y_true"], p["y_pred"], labels=range(K), normalize="true")   # (K, K)
        ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
        ax.set(xticks=range(K), yticks=range(K), xlabel="predicted", ylabel="true", title=f"{rep} ({head})")
        ax.set_xticklabels(classes[dataset], rotation=90, fontsize=7)
        ax.set_yticklabels(classes[dataset], fontsize=7)
        for a in range(K):
            for c in range(K):
                ax.text(c, a, f"{cm[a, c]:.2f}", ha="center", va="center", fontsize=6)
    fig.suptitle(f"{dataset}: confusion matrices (rows normalized)")
    plt.tight_layout()
    plt.savefig(FIG / f"04a_confusion_{dataset}.png", dpi=150)
    plt.show()